In [ ]:
!pip install fuzzywuzzy

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [ ]:
thresholds = [60, 70, 80, 90]

In [ ]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [ ]:
df = df[df["Reality (DeepSeek R1)"] != "hypothetical"]
df = df[df["Co-authors' names (DeepSeek R1)"].notna()]

In [ ]:
len(df)

1355

In [ ]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [ ]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (DeepSeek R1)"].split('/')
    refNames = row["Co-authors' names (Google Scholar)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [ ]:
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral),Match Count 60%,Match Count 70%,Match Count 80%,Match Count 90%
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-,0,0,0,0
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-,0,0,0,0
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-,0,0,0,0
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-,0,0,0,0
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-,0,0,0,0


In [ ]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 80%"] = row["Match Count 80%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 90%"] = row["Match Count 90%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))

/tmp/ipython-input-558499341.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.42105263157894735' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.42105263157894735' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3684210526315789' has dtype incompatible with int64, please explicit

In [ ]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

Mean DNE 0.15480789092268865
Std DNE 0.1726710264474553


In [ ]:
fields = list(df["Field"].unique())
fields

['Agriculture, Fisheries & Forestry',
 'Biology',
 'Built Environment & Design',
 'Clinical Medicine',
 'Earth & Environmental Sciences',
 'Economics & Business',
 'Engineering',
 'Information & Communication Technologies',
 'Mathematics & Statistics',
 'Physics & Astronomy']

In [ ]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [ ]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE without disaggregation

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.11994084162355201
DNE 70% ***** 0.07044189418493178
DNE 80% ***** 0.06111779763372691
DNE 90% ***** 0.05651949205135952
Biology
DNE 60% ***** 0.17630490474331645
DNE 70% ***** 0.10339570073968844
DNE 80% ***** 0.08862103868561608
DNE 90% ***** 0.07904448699282113
Built Environment & Design
DNE 60% ***** 0.1409832835555739
DNE 70% ***** 0.09244922558393867
DNE 80% ***** 0.07763618412871214
DNE 90% ***** 0.0720045540031717
Clinical Medicine
DNE 60% ***** 0.12964835296389746
DNE 70% ***** 0.0634013760912724
DNE 80% ***** 0.05219003982529886
DNE 90% ***** 0.04918507281929665
Earth & Environmental Sciences
DNE 60% ***** 0.1696972931542085
DNE 70% ***** 0.10853699548214911
DNE 80% ***** 0.09197033144368374
DNE 90% ***** 0.07062577845327014
Economics & Business
DNE 60% ***** 0.13453182397450078
DNE 70% ***** 0.0823393089129846
DNE 80% ***** 0.06773058370413688
DNE 90% ***** 0.059667200327808104
Engineering
DNE 60% ***** 0.16132505301695022
DNE

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.18922401218954116
DNE 70% ***** 0.11400340386085675
DNE 80% ***** 0.10133771995165065
DNE 90% ***** 0.07770299171511004
Europe
DNE 60% ***** 0.1598952635499627
DNE 70% ***** 0.10665293655574516
DNE 80% ***** 0.09153196968027706
DNE 90% ***** 0.08535872901548953
Middle East
DNE 60% ***** 0.14325298086886917
DNE 70% ***** 0.0770743143721255
DNE 80% ***** 0.06209471487890787
DNE 90% ***** 0.05599577094203273
North Africa
DNE 60% ***** 0.12285265005177495
DNE 70% ***** 0.05742515503373198
DNE 80% ***** 0.04231390554595782
DNE 90% ***** 0.033278503207024794
North America
DNE 60% ***** 0.15792914527563687
DNE 70% ***** 0.09873671464329253
DNE 80% ***** 0.08714197778909463
DNE 90% ***** 0.08067683616131326
Oceanic
DNE 60% ***** 0.17137055306875962
DNE 70% ***** 0.12245167214526133
DNE 80% ***** 0.1087865623979745
DNE 90% ***** 0.10151696522468633
South/Central America
DNE 60% ***** 0.14869005323223733
DNE 70% ***** 0.08800236128693567
DNE 80% ***** 0.07307

# Mean DNE disaggregated by Citation Level

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.18749455692411046
DNE 70% ***** 0.11250159997289923
DNE 80% ***** 0.09809908923172365
DNE 90% ***** 0.09044786603506935
Low
DNE 60% ***** 0.0428873851088525
DNE 70% ***** 0.0224675422705314
DNE 80% ***** 0.018936011904761903
DNE 90% ***** 0.017819940476190475
Biology
High
DNE 60% ***** 0.23453772702518505
DNE 70% ***** 0.1496660844678054
DNE 80% ***** 0.13037189180823075
DNE 90% ***** 0.11427455507091573
Low
DNE 60% ***** 0.1053336525872891
DNE 70% ***** 0.04700367057104589
DNE 80% ***** 0.037737186442429405
DNE 90% ***** 0.036107841522643305
Built Environment & Design
High
DNE 60% ***** 0.19219258243229356
DNE 70% ***** 0.12847347080939117
DNE 80% ***** 0.11018975882158216
DNE 90% ***** 0.10101934122508273
Low
DNE 60% ***** 0.08245837055360866
DNE 70% ***** 0.05127865961199295
DNE 80% ***** 0.0404320987654321
DNE 90% ***** 0.03884479717813051
Clinical Medicine
High
DNE 60% ***** 0.1541570933286378
DNE 70% ***** 0.08715241560283202

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.25737538773282914
DNE 70% ***** 0.17204568189376873
DNE 80% ***** 0.15587231201049814
DNE 90% ***** 0.12426996429243084
Low
DNE 60% ***** 0.1094151645138487
DNE 70% ***** 0.04603284142757827
DNE 80% ***** 0.03747484240905294
DNE 90% ***** 0.023170615933773828
Europe
High
DNE 60% ***** 0.2287318473606894
DNE 70% ***** 0.1598621085032775
DNE 80% ***** 0.14388953440341565
DNE 90% ***** 0.13490307870816887
Low
DNE 60% ***** 0.07958591577078152
DNE 70% ***** 0.04457556928362409
DNE 80% ***** 0.030448144169948684
DNE 90% ***** 0.02755698770736365
Middle East
High
DNE 60% ***** 0.19927463004908572
DNE 70% ***** 0.11506012467821237
DNE 80% ***** 0.09688878242559137
DNE 90% ***** 0.08536855498927169
Low
DNE 60% ***** 0.08022862554112556
DNE 70% ***** 0.034340277777777775
DNE 80% ***** 0.02295138888888889
DNE 90% ***** 0.02295138888888889
North Africa
High
DNE 60% ***** 0.1778283876590839
DNE 70% ***** 0.09215602989695253
DNE 80% ***** 0.0673276469405022

# Mean Comparison

In [ ]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [ ]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.0884764116119851
highly cited Avg DNE: 0.21176743282735314


In [ ]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [ ]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 14.01778821705573
P-value: 4.365947121862536e-42


## T-Test

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 5.491431333094532e-08
DNE 70% ***** 1.2944267373781786e-05
DNE 80% ***** 4.053208671033243e-05
DNE 90% ***** 0.00011801074572816945
Biology
DNE 60% ***** 3.793594839078939e-06
DNE 70% ***** 9.652581763204711e-06
DNE 80% ***** 3.2147012404975805e-05
DNE 90% ***** 0.000309019528167982
Built Environment & Design
DNE 60% ***** 1.898821883296881e-05
DNE 70% ***** 0.0001289386283406473
DNE 80% ***** 0.00022979299732633945
DNE 90% ***** 0.0006787557461796726
Clinical Medicine
DNE 60% ***** 0.0367730135839991
DNE 70% ***** 0.006870688266820056
DNE 80% ***** 0.001730953150781969
DNE 90% ***** 0.0018313779578011007
Earth & Environmental Sciences
DNE 60% ***** 0.002045326718487057
DNE 70% ***** 0.0025496513250949306
DNE 80% ***** 0.00334339984586683
DNE 90% ***** 0.00010692750137754316
Economics & Business
DNE 60% ***** 1.2146548657704127e-07
DNE 70% ***** 4.605033039838894e-07
DNE 80% ***** 1.1172436244110032e-06
DNE 90% ***** 1.1514675329052113e-0

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 1.2216029649448139e-05
DNE 70% ***** 5.1511538244978835e-06
DNE 80% ***** 6.498992683104629e-06
DNE 90% ***** 6.156830573628725e-06
Europe
DNE 60% ***** 2.3891568634719866e-10
DNE 70% ***** 1.1544184149074766e-09
DNE 80% ***** 4.850764026305453e-10
DNE 90% ***** 2.9949873328741545e-09
Middle East
DNE 60% ***** 8.437962727912386e-07
DNE 70% ***** 2.2121337302471314e-05
DNE 80% ***** 3.317368129905112e-05
DNE 90% ***** 0.0002937298403603105
North Africa
DNE 60% ***** 5.806496981933903e-07
DNE 70% ***** 4.854634894848623e-07
DNE 80% ***** 2.6563414524402823e-05
DNE 90% ***** 0.00019558128589772366
North America
DNE 60% ***** 7.684137528364533e-14
DNE 70% ***** 2.9604952023867758e-12
DNE 80% ***** 1.3184721307316886e-11
DNE 90% ***** 2.4591157129259218e-12
Oceanic
DNE 60% ***** 3.849094695654474e-08
DNE 70% ***** 6.02142003923923e-07
DNE 80% ***** 7.604461159745254e-07
DNE 90% ***** 1.5835870300307752e-06
South/Central America
DNE 60% ***** 0.000269448472

## Mann-Whitney U Test

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 1.0489711633116175e-08
DNE 70% ***** 3.2885228091117393e-07
DNE 80% ***** 5.729582238422657e-07
DNE 90% ***** 1.22935030404261e-06
Biology
DNE 60% ***** 6.048888071527259e-07
DNE 70% ***** 4.7891783798379815e-08
DNE 80% ***** 5.337732294024679e-08
DNE 90% ***** 8.047838579897194e-07
Built Environment & Design
DNE 60% ***** 2.9845566822489065e-06
DNE 70% ***** 3.5855637019146727e-06
DNE 80% ***** 1.013028380992656e-06
DNE 90% ***** 1.3863774017197373e-06
Clinical Medicine
DNE 60% ***** 0.003145394039416753
DNE 70% ***** 0.0008596975728210272
DNE 80% ***** 1.5089245387282382e-05
DNE 90% ***** 2.5176820144695732e-05
Earth & Environmental Sciences
DNE 60% ***** 1.4643936266472415e-05
DNE 70% ***** 9.05926717569721e-08
DNE 80% ***** 8.554956065006093e-09
DNE 90% ***** 3.4032126995996254e-10
Economics & Business
DNE 60% ***** 5.171399402306115e-08
DNE 70% ***** 7.1189180883385155e-09
DNE 80% ***** 3.102365750001915e-09
DNE 90% ***** 2.595532328

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 1.8844539970232593e-07
DNE 70% ***** 9.634960693910297e-09
DNE 80% ***** 1.0602720477494886e-09
DNE 90% ***** 3.337987667524453e-08
Europe
DNE 60% ***** 1.334994908714221e-09
DNE 70% ***** 3.348586772191847e-10
DNE 80% ***** 5.010601303094755e-12
DNE 90% ***** 1.2080800195901134e-11
Middle East
DNE 60% ***** 1.3740084967038695e-08
DNE 70% ***** 4.90296905073866e-09
DNE 80% ***** 4.4516409332368156e-10
DNE 90% ***** 2.421977024880167e-09
North Africa
DNE 60% ***** 1.840971716298982e-06
DNE 70% ***** 3.280153698084522e-08
DNE 80% ***** 1.729443419242689e-07
DNE 90% ***** 4.894213615168985e-07
North America
DNE 60% ***** 8.870917645124541e-15
DNE 70% ***** 1.5507353678448252e-14
DNE 80% ***** 4.24414915727367e-15
DNE 90% ***** 7.173041400262988e-15
Oceanic
DNE 60% ***** 7.8649238320173e-09
DNE 70% ***** 1.1660995158076876e-09
DNE 80% ***** 3.3804102307848474e-10
DNE 90% ***** 2.5193502246872887e-10
South/Central America
DNE 60% ***** 1.873423287731877e-0